In [ ]:
# 1) Install dependencies
!pip install -q --upgrade "transformers>=4.45,<5" "datasets>=2.19,<4" "accelerate>=0.30,<1.14" "bitsandbytes>=0.43,<0.50" "peft>=0.11,<0.19" "sentencepiece>=0.2,<0.3" "huggingface_hub>=0.24,<1.0" "scikit-learn>=1.3,<1.9" "pandas>=2.2,<3.0" "matplotlib>=3.7,<3.10.1" "seaborn>=0.13,<0.14"
print("Dependencies installed.")

In [ ]:
# 2) Configuration
import os
import json
import re
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed, logging as hf_logging
from peft import PeftModel

MODEL_ID = "meta-llama/Meta-Llama-3-8B"
DATASET_PATH = "/kaggle/input/datasets" # Adjust if your dataset is in a subfolder or has a different name
ADAPTER_EVAL_PATH = "/kaggle/input/datasets/llama-8b-trained-model" # Path to your trained LoRA adapter (adjust if needed)

EVAL_MAX_SAMPLES = 450

EVAL_OUTPUT_DIR = "/kaggle/working/eval_outputs"
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

OUTPUT_CATEGORIES = [
    "SIMPLE INSTRUCTION",
    "INSTRUCTION WITH SEQUENCE",
    "PARALLEL INSTRUCTION",
    "INSTRUCTION WITH PURPOSE",
    "INSTRUCTION WITH REASON",
    "EXCLUSIVE INSTRUCTION (OBJECTS)",
    "EXCLUSIVE INSTRUCTION (ACTIONS)",
]

INSTRUCTION = (
    "Classify the instruction type into one of: SIMPLE INSTRUCTION, "
    "INSTRUCTION WITH SEQUENCE, PARALLEL INSTRUCTION, "
    "INSTRUCTION WITH PURPOSE, INSTRUCTION WITH REASON, "
    "EXCLUSIVE INSTRUCTION (OBJECTS), EXCLUSIVE INSTRUCTION (ACTIONS). "
    "Return only one exact label from the list."
)

set_seed(42)
hf_logging.set_verbosity_error()
print("Configuration loaded.")
print(f"Dataset: {DATASET_PATH}")
print(f"Adapter: {ADAPTER_EVAL_PATH}")

In [ ]:
# 3) Authenticate Hugging Face
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = None
for secret_name in ["HF_TOKEN", "META_LLAMA_TOKEN", "HUGGINGFACE_TOKEN", "LLAMA_API"]:
    try:
        HF_TOKEN = UserSecretsClient().get_secret(secret_name)
        if HF_TOKEN:
            print(f"Token loaded from secret: {secret_name}")
            break
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError("Missing HF token in Kaggle Secrets.")

login(token=HF_TOKEN, add_to_git_credential=False)
print("Hugging Face authentication successful.")

In [ ]:
# 4) Load and validate dataset
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

dataset_raw = load_dataset("json", data_files=DATASET_PATH, split="train")
if len(dataset_raw) == 0:
    raise RuntimeError("Dataset is empty.")

required_cols = {"input", "output"}
missing = required_cols - set(dataset_raw.column_names)
if missing:
    raise RuntimeError(f"Missing required columns: {sorted(missing)}")

print(f"Total rows: {len(dataset_raw):,}")
print("Label distribution:")
for label, count in sorted(Counter(dataset_raw["output"]).items(), key=lambda x: -x[1]):
    print(f"  {label}: {count}")

In [ ]:
# 5) Load base model and LoRA adapter
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this model.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
 )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token
elif tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
 )

if not os.path.isdir(ADAPTER_EVAL_PATH):
    raise FileNotFoundError(f"Adapter path not found: {ADAPTER_EVAL_PATH}")

model = PeftModel.from_pretrained(base_model, ADAPTER_EVAL_PATH, is_trainable=False)
model.eval()
print("Model and adapter loaded.")

In [ ]:
# 6) Evaluate metrics + confusion matrix heatmap
def normalize_label(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.upper()

label_norm_to_original = {normalize_label(x): x for x in OUTPUT_CATEGORIES}
eval_labels = list(label_norm_to_original.keys())

def extract_pred_label(generated_text):
    first_line = generated_text.strip().split("\n")[0].strip()
    cand = normalize_label(first_line)
    if cand in label_norm_to_original:
        return cand
    matches = [lbl for lbl in eval_labels if lbl in normalize_label(generated_text)]
    if len(matches) == 1:
        return matches[0]
    return "UNKNOWN"

if EVAL_MAX_SAMPLES is None:
    records = dataset_raw
else:
    records = dataset_raw.select(range(min(EVAL_MAX_SAMPLES, len(dataset_raw))))

y_true, y_pred, mistakes = [], [], []
model.eval()

for ex in records:
    inp = str(ex.get("input", ""))
    true_label = normalize_label(str(ex.get("output", "")))

    prompt = f"### Instruction:\n{INSTRUCTION}\n\n### Input:\n{inp}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=20,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    generated = tokenizer.decode(gen_ids, skip_special_tokens=True)
    pred_label = extract_pred_label(generated)

    gt = true_label if true_label in eval_labels else "UNKNOWN"
    y_true.append(gt)
    y_pred.append(pred_label)

    if gt != pred_label:
        mistakes.append({
            "input": inp,
            "expected": gt,
            "predicted": pred_label,
            "raw_generation": generated,
        })

acc = accuracy_score(y_true, y_pred)
full_labels = eval_labels + (["UNKNOWN"] if ("UNKNOWN" in y_true or "UNKNOWN" in y_pred) else [])

print(f"Evaluated samples: {len(y_true)}")
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print("\nClassification report:")
print(classification_report(y_true, y_pred, labels=full_labels, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=full_labels)
cm_df = pd.DataFrame(cm, index=full_labels, columns=full_labels)
display(cm_df)

plt.figure(figsize=(1.4 * len(full_labels), 1.1 * len(full_labels)))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="YlGnBu", cbar=True, linewidths=0.5, linecolor="white")
plt.title("Confusion Matrix Heatmap")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

heatmap_path = os.path.join(EVAL_OUTPUT_DIR, "confusion_matrix_heatmap.png")
plt.savefig(heatmap_path, dpi=220, bbox_inches="tight")
plt.show()

mistakes_df = pd.DataFrame(mistakes)
if len(mistakes_df) > 0:
    display(mistakes_df.head(30))

with open(os.path.join(EVAL_OUTPUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump({"samples": len(y_true), "accuracy": float(acc), "labels": full_labels}, f, indent=2)
cm_df.to_csv(os.path.join(EVAL_OUTPUT_DIR, "confusion_matrix.csv"), index=True)
mistakes_df.to_csv(os.path.join(EVAL_OUTPUT_DIR, "misclassified_samples.csv"), index=False)
print(f"Saved artifacts in: {EVAL_OUTPUT_DIR}")

In [ ]:
# 7) Zip outputs for download
import shutil

zip_base = "/kaggle/working/eval_outputs_bundle"
zip_file = f"{zip_base}.zip"
if os.path.exists(zip_file):
    os.remove(zip_file)

shutil.make_archive(zip_base, "zip", EVAL_OUTPUT_DIR)
print(f"Created zip: {zip_file}")
print(f"Zip size: {os.path.getsize(zip_file) / 1e6:.2f} MB")
print("Download from Kaggle Output panel after Save Version with Save Output enabled.")

In [ ]:
!pip install -q evaluate bert-score
import evaluate

references = [str(x) for x in y_true]
predictions = [str(x) for x in y_pred]

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

result_rouge = rouge.compute(predictions=predictions, references=references)
result_bert = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en"
)

print("ROUGE-L:", result_rouge["rougeL"])
print("Average BERTScore F1:", sum(result_bert["f1"]) / len(result_bert["f1"]))

In [ ]:
# 8) Package evaluation outputs for download
import os
import shutil

zip_base = "/kaggle/working/eval_outputs_bundle"
zip_file = f"{zip_base}.zip"

if not os.path.isdir(EVAL_OUTPUT_DIR):
    raise FileNotFoundError(
        f"Evaluation output folder not found: {EVAL_OUTPUT_DIR}. "
        "Run the evaluation cell first."
    )

# Create a fresh zip each run
if os.path.exists(zip_file):
    os.remove(zip_file)

shutil.make_archive(zip_base, "zip", EVAL_OUTPUT_DIR)
print(f"Created zip: {zip_file}")
print(f"Zip size: {os.path.getsize(zip_file) / 1e6:.2f} MB")

print("\nDownload steps in Kaggle:")
print("1. Right panel -> Output")
print("2. Find eval_outputs_bundle.zip")
print("3. Click to download")
print("4. If not visible, click Save Version with Save Output enabled")